## Importamos librerias

In [2]:
%pip install tinydb
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer 
from tinydb import TinyDB
from datetime import datetime

  Using cached tinydb-4.8.2-py3-none-any.whl.metadata (6.7 kB)
Using cached tinydb-4.8.2-py3-none-any.whl (24 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



### A. CONFIGURACIÓN DE LA BASE DE DATOS NOSQL

 Creamos la base de datos (archivo JSON en carpeta)

In [3]:
# Creamos la base de datos (archivo JSON en tu carpeta)
db = TinyDB('base_nosql_siniestros.json')

# Creamos las 3 colecciones (tablas) que exige la consigna
tabla_datos = db.table('datos_entrada')
tabla_resultados = db.table('resultados_modelo')
tabla_config = db.table('configuracion_modelo')

# Limpiamos las tablas antes de correr para no duplicar datos cada vez que ejecutes la celda
#tabla_datos.truncate()
#tabla_resultados.truncate()
#tabla_config.truncate()

### B. PREPARACIÓN DE DATOS

In [4]:
# 1. Cargar los datos limpios
X_train = pd.read_csv('X_train_limpio.csv')
y_train = pd.read_csv('y_train_limpio.csv')
X_test = pd.read_csv('X_test_limpio.csv')
y_test = pd.read_csv('y_test_limpio.csv')

# Solución de índices duplicados
y_train = y_train.iloc[:, -1:]
y_test = y_test.iloc[:, -1:]

# 2. One-Hot Encoding y rellenar NaNs
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

imputador = SimpleImputer(strategy='mean')
X_train = pd.DataFrame(imputador.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(imputador.transform(X_test), columns=X_test.columns)

imputador_y = SimpleImputer(strategy='mean')
y_train = pd.DataFrame(imputador_y.fit_transform(y_train), columns=y_train.columns)
y_test = pd.DataFrame(imputador_y.transform(y_test), columns=y_test.columns)

y_train_arr = y_train.values.ravel()
y_test_arr = y_test.values.ravel()

### C. GUARDAR DATOS DE ENTRADA (NOSQL)

In [5]:
print("💾 Guardando datos de entrada en NoSQL...")
# Unimos X e y de prueba para guardar el dataset. 
# Nota: Guardamos una muestra representativa (100 filas) para que el archivo JSON no colapse la PC.
datos_guardar = X_test.copy()
datos_guardar['Variable_Objetivo_Real'] = y_test_arr
tabla_datos.insert_multiple(datos_guardar.head(100).to_dict(orient='records'))

💾 Guardando datos de entrada en NoSQL...


[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100]

### D. ENTRENAMIENTO Y GUARDADO DE CONFIGURACIÓN (NOSQL)

In [10]:
modelo_lineal = LinearRegression()
modelo_rf = RandomForestRegressor(random_state=100)

print("\n⚙️ Guardando configuración de modelos en NoSQL...")
# Guardamos los hiperparámetros (parametrización) de ambos modelos
tabla_config.insert({
    "fecha": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "modelo": "Regresión Lineal",
    "hiperparametros": modelo_lineal.get_params()
})
tabla_config.insert({
    "fecha": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "modelo": "Random Forest",
    "hiperparametros": modelo_rf.get_params()
})

print("Entrenando modelos...")
modelo_lineal.fit(X_train, y_train_arr)
modelo_rf.fit(X_train, y_train_arr)

predicciones_lineal = modelo_lineal.predict(X_test)
predicciones_rf = modelo_rf.predict(X_test)


⚙️ Guardando configuración de modelos en NoSQL...
Entrenando modelos...


### E. EVALUACIÓN Y GUARDADO DE RESULTADOS (NOSQL)

In [11]:
def evaluar_y_mostrar_metricas(y_real, y_predicha, nombre_del_modelo):
    # 1. Cálculos
    rmse = np.sqrt(mean_squared_error(y_real, y_predicha))
    mae = mean_absolute_error(y_real, y_predicha)
    r2 = r2_score(y_real, y_predicha)
    
    # 2. Imprimir Métricas en pantalla
    print(f"\n--- Resultados para {nombre_del_modelo} ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R²:   {r2:.4f}")
    
    # 3. Imprimir Tabla comparativa en pantalla
    print("\nMuestra de predicciones (Real vs Predicho):")
    tabla_comparativa = pd.DataFrame({'Valor Real': y_real, 'Predicción': np.round(y_predicha, 2)})
    print(tabla_comparativa.head(5).to_string(index=False))
    print("=" * 40)
    
    # 4. GUARDAR RESULTADOS EN NOSQL
    # Guardamos las métricas y una pequeña muestra de las predicciones para auditoría
    predicciones_muestra = [{"real": float(r), "predicho": float(p)} for r, p in zip(y_real[:5], y_predicha[:5])]
    
    tabla_resultados.insert({
        "fecha": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "modelo": nombre_del_modelo,
        "metricas": {"rmse": float(rmse), "mae": float(mae), "r2": float(r2)},
        "muestra_predicciones": predicciones_muestra
    })

print("\n📊 Evaluando y guardando resultados...")
evaluar_y_mostrar_metricas(y_test_arr, predicciones_lineal, "Regresión Lineal")
evaluar_y_mostrar_metricas(y_test_arr, predicciones_rf, "Random Forest")
print("\n✅ ¡Todos los requerimientos de la base NoSQL completados!")


📊 Evaluando y guardando resultados...

--- Resultados para Regresión Lineal ---
RMSE: 1.2529
MAE:  0.6169
R²:   0.0432

Muestra de predicciones (Real vs Predicho):
 Valor Real  Predicción
        1.0        1.25
        2.0        1.64
        2.0        2.10
        2.0        1.61
        1.0        1.37

--- Resultados para Random Forest ---
RMSE: 0.9013
MAE:  0.4972
R²:   0.5049

Muestra de predicciones (Real vs Predicho):
 Valor Real  Predicción
        1.0        1.11
        2.0        1.65
        2.0        1.52
        2.0        1.41
        1.0        1.30

✅ ¡Todos los requerimientos de la base NoSQL completados!


¿Qué hace este código para cumplir la consigna?

    Tabla de datos de entrada: Toma tus datos de prueba limpios (X_test + la variable objetivo) y guarda los primeros 100 registros en la colección datos_entrada. (Guardamos 100 para no hacer un archivo local de cientos de megas, lo cual es totalmente válido académicamente).

    Tabla de configuración/parametrización: Usa la función nativa .get_params() de scikit-learn para extraer todos los parámetros matemáticos con los que se configuró cada algoritmo y los guarda en configuracion_modelo.

    Tabla de resultados del modelo: Guarda el nombre del modelo, el RMSE, MAE, R² y una pequeña lista de Real vs Predicho adentro de resultados_modelo.

Cuando lo ejecutes, vas a ver que en tu carpeta de Windows (en tu Escritorio) se va a crear un archivo llamado base_nosql_siniestros.json